# 03 — Main Text Figures

Produces **only** the figures that appear in the main text of the manuscript. Reads pre-fitted `.pkl` artefacts from `01_analysis.ipynb`, `02_simulation.ipynb`, and the three formal-model notebooks; no refitting happens here.

For appendix figures, see `03_appendix_text_figures.ipynb`. For presentation exports, see `04_presentation_figures.ipynb`.

**Figure list:**
1. **Figure 1** — Mechanistic-model predictions across the three variants (Ecology of Contagions Model / No Interaction Model / General Arousal Model), five panels per variant (evaluative-space trajectories, hazard decay, interaction windows, contagiousness–potency tradeoff, settler dynamics).
2. **Figure 2** — Exposure hazard ratios for first adoption vs second+ adoptions (both stratified).
3. **Figure 3** — Temporal hazard shift: ratio of each Weibull model's instantaneous baseline hazard to Model 1's constant baseline at 1 h, 24 h, and 72 h after the most recent prior adoption.
4. **Figure 4** — Decay-to-baseline time $t^*$ for each sequential Weibull model, showing how the interaction window expands with adoption depth.
5. **Figure 5** — Settler-effect pattern across three clusterings (semantic, first-appearance, peak-frequency).
6. **Figure 6** — 2D gateway scatter: per-conspiracy Model 1 HR (contagiousness) vs Model 2b HR (gateway acceleration), relative to `fakenews`.
7. **Figure 7** — Counterfactual intervention effects (quarantine, shadow banning, reputation nudge): top row shows total % AUC change vs baseline, bottom row shows efficiency (% AUC reduction per unit intensity).

In [ ]:
# ── Imports & publication style ──────────────────────────────────
%matplotlib inline
import sys, os, joblib
sys.path.insert(0, '..')

import numpy as np
import matplotlib.pyplot as plt

from conspiracy_analysis.visualization import (
    set_nhb_style,
    NHB_COL_SINGLE, NHB_COL_ONE_HALF, NHB_COL_DOUBLE,
    NHB_PALETTE, CLUSTER_PALETTE,
    apply_panel_label, nhb_annotation_fontsize,
)
from conspiracy_analysis.simulation.provenance import load_manifest_backed_simulation_bundle
set_nhb_style()

# Output directories for paper and optional figure renders.
FIGURE_FINAL_DIR = os.path.join('..', 'figures_final')
FIGURE_OPTIONAL_DIR = os.path.join('..', 'figures_optional')
os.makedirs(FIGURE_FINAL_DIR, exist_ok=True)
os.makedirs(FIGURE_OPTIONAL_DIR, exist_ok=True)
print(f'Paper figures will be saved to: {os.path.abspath(FIGURE_FINAL_DIR)}')
print(f'Optional figures will be saved to: {os.path.abspath(FIGURE_OPTIONAL_DIR)}')

SIM_BOOTSTRAP_N = 2000
SIM_BOOTSTRAP_SEED = 20240517

def bootstrap_mean_ci(values, n_boot=SIM_BOOTSTRAP_N, seed=SIM_BOOTSTRAP_SEED):
    values = np.asarray(values, dtype=float)
    values = values[~np.isnan(values)]
    estimate = float(np.mean(values)) if values.size else float('nan')
    if values.size == 0:
        return estimate, float('nan'), float('nan')
    rng = np.random.default_rng(seed)
    idx = rng.integers(0, values.size, size=(n_boot, values.size))
    boot = values[idx].mean(axis=1)
    lower, upper = np.percentile(boot, [2.5, 97.5])
    return estimate, float(lower), float(upper)

def bootstrap_auc_change_ci(
    baseline_values,
    scenario_values,
    intensity=None,
    n_boot=SIM_BOOTSTRAP_N,
    seed=SIM_BOOTSTRAP_SEED,
):
    baseline_values = np.asarray(baseline_values, dtype=float)
    scenario_values = np.asarray(scenario_values, dtype=float)
    if baseline_values.shape != scenario_values.shape:
        raise ValueError('Baseline and scenario arrays must have matching shapes.')
    if baseline_values.size == 0:
        return {
            'change': float('nan'),
            'change_ci': (float('nan'), float('nan')),
            'efficiency': float('nan'),
            'efficiency_ci': (float('nan'), float('nan')),
        }
    baseline_mean = float(np.mean(baseline_values))
    scenario_mean = float(np.mean(scenario_values))
    change = (scenario_mean / baseline_mean - 1.0) * 100.0
    rng = np.random.default_rng(seed)
    idx = rng.integers(0, baseline_values.size, size=(n_boot, baseline_values.size))
    boot_baseline = baseline_values[idx].mean(axis=1)
    boot_scenario = scenario_values[idx].mean(axis=1)
    boot_change = (boot_scenario / boot_baseline - 1.0) * 100.0
    change_lower, change_upper = np.percentile(boot_change, [2.5, 97.5])
    result = {
        'change': change,
        'change_ci': (float(change_lower), float(change_upper)),
    }
    if intensity is not None:
        efficiency = -change / intensity
        boot_efficiency = -boot_change / intensity
        eff_lower, eff_upper = np.percentile(boot_efficiency, [2.5, 97.5])
        result.update({
            'efficiency': efficiency,
            'efficiency_ci': (float(eff_lower), float(eff_upper)),
        })
    return result

def ci_error_array(values, lowers, uppers):
    values = np.asarray(values, dtype=float)
    lowers = np.asarray(lowers, dtype=float)
    uppers = np.asarray(uppers, dtype=float)
    invalid = (lowers > values) | (uppers < values)
    if np.any(invalid):
        raise ValueError('CI bounds must bracket point estimates.')
    return np.vstack([
        values - lowers,
        uppers - values,
    ])

In [ ]:
from pathlib import Path

INTERMEDIATE_DIR = Path("../intermediate_files")
INTERMEDIATE_DIR.mkdir(exist_ok=True)

def intermediate_path(filename):
    return INTERMEDIATE_DIR / filename


In [ ]:
# ── Load pre-fitted Cox models from .pkl files ────────────────────────
# Dummy-coded models (per-conspiracy coefficients): model_1 .. model_8
model_names = [
    'model_1', 'model_2a', 'model_2b',
    'model_3', 'model_4', 'model_5',
    'model_6', 'model_7', 'model_8',
]
cox_results = {}
for name in model_names:
    pkl_path = intermediate_path(f'{name}_cox.pkl')
    if os.path.exists(pkl_path):
        ctv = joblib.load(pkl_path)
        cox_results[name] = (ctv, None)
        print(f'Loaded {pkl_path}: {len(ctv.params_)} parameters')
    else:
        print(f'WARNING: {pkl_path} not found — run 01_analysis.ipynb first')

# Stratified variants used for Figure 1c (Model 1 vs Pooled, matching strata).
stratified_models = {}
for name in ['model_1_strat', 'model_pooled']:
    pkl_path = intermediate_path(f'{name}_cox.pkl')
    if os.path.exists(pkl_path):
        stratified_models[name] = joblib.load(pkl_path)
        print(f'Loaded {pkl_path}: {len(stratified_models[name].params_)} parameters')
    else:
        print(f'WARNING: {pkl_path} not found — run 01_analysis.ipynb first')

# Parametrize baseline hazards for Figures 4 / 5.
from conspiracy_analysis.models.baseline_hazards import (
    parametrize_all_baselines, compute_all_decay_times,
)
from conspiracy_analysis.models.bootstrap_inference import load_bootstrap_artifact
cox_models_dict = {n: ctv for n, (ctv, _) in cox_results.items() if n != 'model_2b'}
all_baselines = parametrize_all_baselines(cox_models_dict)
decay_times = compute_all_decay_times(all_baselines)
print(f'\nParametrized {len(all_baselines)} baseline hazards')
cox_bootstrap_results = load_bootstrap_artifact(intermediate_path('cox_bootstrap_results.pkl'))
timeline_bootstrap_results = load_bootstrap_artifact(intermediate_path('timeline_bootstrap_results.pkl'))
print('Loaded bootstrap interval artifacts')


In [ ]:
# ── Load graph + semantic matrix + clustering + barrier/settler structures ──────
import networkx as nx
from conspiracy_analysis import BOT_SCORE_THRESHOLD
from conspiracy_analysis.data.loader import load_graph_and_tweets
from conspiracy_analysis.config import (
    apply_conspiracy_config_to_graph,
    get_baseline_conspiracy,
    get_included_conspiracies,
)
from conspiracy_analysis.analysis.semantic import (
    find_optimal_clustering, assign_clusters, load_clustering_result,
    compute_temporal_distance_matrix, compute_peak_frequency_distance_matrix,
)
from conspiracy_analysis.analysis.statistics import (
    extract_user_timelines, compute_semantic_barrier_analysis, compute_settler_effect,
)

G, _ = load_graph_and_tweets(from_joblib=True, time_resolution='Hour')
G = G.to_undirected()
G.remove_nodes_from(list(nx.isolates(G)))

# Reference and analytic categories come from config/conspiracies.yaml.
BASELINE_CONSPIRACY = get_baseline_conspiracy()
INCLUDED_CONSPIRACIES = get_included_conspiracies()
G = apply_conspiracy_config_to_graph(G)
conspiracies = G.graph['conspiracy_cols']

# Semantic clustering saved by 01_analysis.ipynb.
clustering_result = load_clustering_result()
cluster_assignments = assign_clusters(clustering_result)

# Temporal (first-appearance) clustering.
df_temporal = compute_temporal_distance_matrix(G)
temporal_clustering = find_optimal_clustering(
    df_temporal, conspiracies,
    methods=['ward', 'average', 'complete', 'single'], k_range=range(3, 9),
)
temporal_cluster_assignments = assign_clusters(temporal_clustering)

# Peak-frequency (24h window) clustering.
df_peak_freq = compute_peak_frequency_distance_matrix(G, window=24)
peak_freq_clustering = find_optimal_clustering(
    df_peak_freq, conspiracies,
    methods=['ward', 'average', 'complete', 'single'], k_range=range(3, 9),
)
peak_freq_cluster_assignments = assign_clusters(peak_freq_clustering)

# Semantic barrier + settler.
user_timelines = extract_user_timelines(
    G, cluster_assignments, bot_score_threshold=BOT_SCORE_THRESHOLD,
    mode='HUMAN', min_conspiracies=2,
)
semantic_gaps = compute_semantic_barrier_analysis(user_timelines, cluster_assignments)
semantic_settler = compute_settler_effect(user_timelines, cluster_assignments)

# Temporal (first-appearance) barrier + settler.
temporal_timelines = extract_user_timelines(
    G, temporal_cluster_assignments, bot_score_threshold=BOT_SCORE_THRESHOLD,
    mode='HUMAN', min_conspiracies=2,
)
temporal_gaps = compute_semantic_barrier_analysis(temporal_timelines, temporal_cluster_assignments)
temporal_settler = compute_settler_effect(temporal_timelines, temporal_cluster_assignments)

# Peak-frequency barrier + settler.
peak_freq_timelines = extract_user_timelines(
    G, peak_freq_cluster_assignments, bot_score_threshold=BOT_SCORE_THRESHOLD,
    mode='HUMAN', min_conspiracies=2,
)
peak_freq_gaps = compute_semantic_barrier_analysis(peak_freq_timelines, peak_freq_cluster_assignments)
peak_freq_settler = compute_settler_effect(peak_freq_timelines, peak_freq_cluster_assignments)

print(f'\nIncluded conspiracies: {len(conspiracies)}')
print(f'Semantic clustering: {clustering_result["best_method"]} linkage, k={clustering_result["best_k"]}, '
      f'silhouette={clustering_result["best_score"]:.4f}')
print(f'Temporal (first-appearance) clustering: {temporal_clustering["best_method"]} linkage, '
      f'k={temporal_clustering["best_k"]}, silhouette={temporal_clustering["best_score"]:.4f}')
print(f'Peak-frequency clustering: {peak_freq_clustering["best_method"]} linkage, '
      f'k={peak_freq_clustering["best_k"]}, silhouette={peak_freq_clustering["best_score"]:.4f}')


In [ ]:
# ── Lazy simulation bundle loader ──────────────────────────────────
# Uncomment once a main-text figure needs counterfactual / ABM outputs.
# The bundle is ~320 MB, so we don't load it unless required.
#
# bundle = joblib.load(intermediate_path('simulation_results_all.pkl'))
# cfg = bundle['config']
# counterfactual_steps = cfg['counterfactual_steps']
# print('Loaded simulation_results_all.pkl')

---

## Figure 1 — Mechanistic-Model Predictions (3 Variants × 5 Panels)

Composite figure: one column per model variant, five rows of panels (a–e). Each variant operationalizes a distinct candidate mechanism for sequential conspiracy adoption (all three share the same structural framework and differ only in two parameters, $\alpha$ and $\beta$):

- **Ecology of Contagions Model** ($\alpha > 0$, $\beta = 0$) — `ecology_of_contagions_model.ipynb`. Adoption shifts the user's orientation toward the adopted narrative's position (content-aligned reshaping); the mechanism we call cross-facilitation.
- **No Interaction Model** ($\alpha = 0$, $\beta = 0$) — `no_interaction_model.ipynb`. State never moves; sequential adoptions are independent of each other.
- **General Arousal Model** ($\alpha = 0$, $\beta > 0$) — `general_arousal_model.ipynb`. State never moves, but each adoption produces a transient, content-agnostic elevation of the hazard for all narratives.

Panels (rows, top to bottom):

- **a** — Adoption trajectories in the 2D evaluative space (first 5 users).
- **b** — Mean hazard ratio decay curves by adoption depth.
- **c** — Interaction window duration by adoption depth (time to 1.1× never-adopted baseline).
- **d** — Contagiousness–potency tradeoff relative to reference narrative A1.
- **e** — Settler dynamics (median transition times).

In [ ]:
# Figure 1 — Formal Model Results (3 variants × 5 panels).
# Reuses panel plotters from conspiracy_analysis.visualization.formal_model_panels.
import joblib
from conspiracy_analysis.visualization.formal_model_panels import PANELS
from conspiracy_analysis.visualization.style import apply_panel_label

# Formal model notebooks save these pickle bundles to intermediate_files.
# (label, pkl, reshaping_active). reshaping_active=False marks variants where
# narrative signatures are static (alpha=0), which row (a) annotates in-axes.
# Labels match the paper's variant names (Section 2.2).
model_specs = [
    ('Ecology of Contagions Model', 'ecology_of_contagions_model_results.pkl',       True),
    ('No Interaction Model',   'no_interaction_model_results.pkl',  False),
    ('General Arousal Model',  'general_arousal_model_results.pkl', False),
]

model_results = {}
for label, fname, _ in model_specs:
    path = intermediate_path(fname)
    if not os.path.exists(path):
        raise FileNotFoundError(
            f'Missing {path}. Run Formal model/{fname.replace("_results.pkl",".ipynb")} first to regenerate the intermediate_files artifact.'
        )
    model_results[label] = joblib.load(path)
    print(f'Loaded {fname}')

# 5 rows (panels a-e) x 3 columns (models). Sized roughly for full NHB double width,
# scaled up to keep individual panels legible.
n_rows, n_cols = len(PANELS), len(model_specs)
fig, axes = plt.subplots(
    n_rows, n_cols,
    figsize=(NHB_COL_DOUBLE[0] * 1.15, NHB_COL_DOUBLE[1] * 2.2),
    squeeze=False,
)

for row, (letter, _, plotter) in enumerate(PANELS):
    for col, (label, _, reshaping_active) in enumerate(model_specs):
        ax = axes[row, col]
        # Row (a) is special-cased: legend only on column 0; models without
        # belief reshaping get an in-axes note so the static positions are obvious.
        if letter == 'a':
            # Full-interaction column (col 0) shows a single user's trajectory so the
            # reshaping dynamics are legible; the other two variants (static positions)
            # still overlay the default 5 for context.
            plotter(
                ax, model_results[label],
                show_legend=(col == 0),
                no_reshaping_note=not reshaping_active,
                max_users=(1 if col == 0 else 5),
            )
        elif letter == 'd' and col == 0:
            # Full-interaction tradeoff: widen y-axis so the 'reference (A1)'
            # label doesn't collide with the bottom axis.
            plotter(ax, model_results[label], ymin=0.8)
        else:
            plotter(ax, model_results[label])
        # Column title only on the top row.
        if row == 0:
            ax.set_title(label)
        # Panel letter only on the leftmost column.
        if col == 0:
            apply_panel_label(ax, letter)
        # Keep one y-axis label per row (leftmost column); suppress it on the rest.
        if col > 0:
            ax.set_ylabel('')

# Row (d) — harmonise y-limits across all three columns using the Full
# interaction column's limits (col 0 was plotted with ymin=0.8, giving the
# reference-label clearance). This makes the tradeoff structures directly
# comparable across variants.
d_row_idx = [i for i, (letter, *_rest) in enumerate(PANELS) if letter == 'd'][0]
d_ylim = axes[d_row_idx, 0].get_ylim()
for col in range(1, n_cols):
    axes[d_row_idx, col].set_ylim(d_ylim)

fig.tight_layout()
# Align y-axis labels across the leftmost column at a fixed axes-coordinate x,
# pulling them closer to the spine than align_ylabels' default (which anchors to
# the widest tick label in the column).
for ax_left in axes[:, 0]:
    ax_left.yaxis.set_label_coords(-0.11, 0.5)
out_path = os.path.join(FIGURE_FINAL_DIR, 'fig01_formal_models.png')
fig.savefig(out_path, bbox_inches='tight')
plt.show()
print(f'Saved: {out_path}')

---

## Figure 2 — Exposure Hazard Ratios: First Adoption vs Second+ Adoptions

Re-references each model's exposure HRs to `s7_d1` (1 neighbor) so both curves sit on a common scale. Both models use stratified Cox specifications:

- **First Adoption**: stratified by `conspiracy` (per-conspiracy baseline hazards).
- **Second+ Adoptions**: stratified by `(adoption_order, conspiracy)` (per-stage per-conspiracy baselines).

Stratification details belong in the figure caption, not the legend.

In [ ]:
# Figure 2 — Exposure HRs: First Adoption vs Second+ Adoptions (both stratified).
# The dict keys become the legend labels (plot_exposure_hr_comparison uses them verbatim
# when they are not in the built-in _MODEL_LABELS map).
from conspiracy_analysis.visualization.figures import plot_exposure_hr_comparison

cox_fig1 = {
    'First Conspiracy Theory Adoption': (stratified_models['model_1_strat'], None),
    'Second+ Conspiracy Theory Adoption': (stratified_models['model_pooled'], None),
}

fig1 = plot_exposure_hr_comparison(
    cox_fig1,
    models_to_include=list(cox_fig1.keys()),
    figsize=NHB_COL_ONE_HALF,
    save_path=os.path.join(FIGURE_OPTIONAL_DIR, 'fig02_exposure_hr_first_vs_pooled.png'),
    bootstrap_intervals=cox_bootstrap_results,
    bootstrap_model_map={
        'First Conspiracy Theory Adoption': 'model_1_strat',
        'Second+ Conspiracy Theory Adoption': 'model_pooled',
    },
)
plt.show()
print(f'Saved: {os.path.join(FIGURE_OPTIONAL_DIR, "fig02_exposure_hr_first_vs_pooled.png")}')

---

## Figure 3 — Temporal Hazard Shift

Ratio of each Weibull model's instantaneous baseline hazard to Model 1's constant baseline, evaluated at three time points after the most recent prior adoption: 1 hour, 24 hours, and 72 hours. The dashed grey line at 1.0 marks Model 1's constant baseline. The hazard elevation grows with adoption depth across all three time horizons, showing that cross-facilitation compounds with each successive adoption.

In [ ]:
# Figure 3 — Temporal hazard shift (1 h, 24 h, 72 h).
# Reuses the helper from conspiracy_analysis.visualization.figures; inputs come
# from the baseline-hazard parametrization already computed in the Cox loader cell.
from conspiracy_analysis.visualization.figures import plot_temporal_hazard_shift

fig3 = plot_temporal_hazard_shift(
    all_baselines,
    eval_times_hours=[1.0, 24.0, 72.0],
    figsize=(NHB_COL_ONE_HALF[0], NHB_COL_DOUBLE[1] * 0.65),
    save_path=os.path.join(FIGURE_OPTIONAL_DIR, 'fig03_temporal_hazard_shift.png'),
    bootstrap_intervals=cox_bootstrap_results,
)
plt.show()
print(f'Saved: {os.path.join(FIGURE_OPTIONAL_DIR, "fig03_temporal_hazard_shift.png")}')

---

## Figure 4 — Decay-to-Baseline Time by Adoption Depth

Analytical decay time $t^* = \lambda \cdot (a\lambda/k)^{1/(k-1)}$, the time at which each Weibull model's instantaneous baseline hazard decays back to Model 1's constant baseline $a$. $t^*$ thus quantifies the duration of the elevated-susceptibility window following each adoption. Values expand with adoption depth.

In [ ]:
# Figure 4 — Decay-to-baseline times $t^*$ by adoption depth.
# Reuses plot_decay_times_line from conspiracy_analysis.visualization.plots;
# `decay_times` was computed in the Cox loader cell.
from conspiracy_analysis.visualization.plots import plot_decay_times_line

fig4 = plot_decay_times_line(
    decay_times,
    figsize=(NHB_COL_ONE_HALF[0], NHB_COL_DOUBLE[1] * 0.65),
    save_path=os.path.join(FIGURE_OPTIONAL_DIR, 'fig04_decay_times_line.png'),
    bootstrap_intervals=cox_bootstrap_results,
)
plt.show()
print(f'Saved: {os.path.join(FIGURE_OPTIONAL_DIR, "fig04_decay_times_line.png")}')

---

## Figure 5 — Settler Effect Across Three Clusterings

Three columns, one row. Each panel shows the settler-effect pattern — median hours to the next adoption in three phases (within old cluster → jump to new cluster → sharing additional in new cluster) — under a different clustering of the 15 conspiracies:

- **Semantic clustering** — clusters derived from the SBERT-based narrative-similarity matrix.
- **Temporal (first-appearance) clustering** — clusters derived from pairwise distances between conspiracies' first-appearance times in the data.
- **Peak-frequency clustering** — clusters derived from pairwise distances between each conspiracy's 24-hour peak-frequency time.

A common y-axis makes the magnitude of the "jump" comparable across clusterings.

In [ ]:
# Figure 5 — Settler effect under three clusterings (semantic, first-appearance, peak-frequency).
# Reuses plot_settler_effect_triptych from conspiracy_analysis.visualization.figures.
# show_significance=True overlays Holm-corrected pairwise Wilcoxon brackets
# (* p<0.05, ** p<0.01, *** p<0.001, n.s. otherwise).
import importlib
from conspiracy_analysis.visualization import figures as figure_module

figure_module = importlib.reload(figure_module)
plot_settler_effect_triptych = figure_module.plot_settler_effect_triptych

fig5 = plot_settler_effect_triptych(
    [
        ('Semantic clustering',   semantic_settler),
        ('First-appearance clustering', temporal_settler),
        ('Peak-frequency clustering',   peak_freq_settler),
    ],
    figsize=(NHB_COL_DOUBLE[0] * 1.1, NHB_COL_DOUBLE[1] * 0.8),
    show_significance=True,
    save_path=os.path.join(FIGURE_OPTIONAL_DIR, 'fig05_settler_effect_triptych.png'),
    bootstrap_intervals=timeline_bootstrap_results,
    bootstrap_family_map={
        'Semantic clustering': 'semantic',
        'First-appearance clustering': 'temporal',
        'Peak-frequency clustering': 'peak_frequency',
    },
)
plt.show()
print(f'Saved: {os.path.join(FIGURE_OPTIONAL_DIR, "fig05_settler_effect_triptych.png")}')

---

## Figure 6 — 2D Gateway Scatter

Each conspiracy is plotted on two hazard-ratio axes, both relative to the reference conspiracy *fakenews*:

- **X-axis (contagiousness)** — Model 1 HR: how much more likely a user is to adopt this conspiracy first compared to *fakenews*.
- **Y-axis (gateway acceleration)** — Model 2b HR: how much adopting this conspiracy first accelerates a user's *second* adoption (any other conspiracy) compared to adopting *fakenews* first.

Points in the top-right quadrant (HR > 1 on both axes) are "gateways": they are both adopted often and, once adopted, strongly accelerate downstream adoption. Dashed grey reference lines mark HR = 1 on both axes; *fakenews* sits at (1, 1) by construction. Colour groups each point into one of four semantic clusters.


In [ ]:
# Figure 6 — 2D Gateway Scatter.
# identify_gateway_2d pulls the per-conspiracy HR + CI from Model 1 (contagiousness)
# and Model 2b (gateway acceleration); plot_gateway_scatter renders them on orthogonal
# HR axes with semantic cluster colour groups and a reference marker at (1, 1).
from conspiracy_analysis.models.gateway_effects import identify_gateway_2d
from conspiracy_analysis.visualization.plots import plot_gateway_scatter

gateway_2d = identify_gateway_2d(
    cox_results['model_1'][0],
    cox_results['model_2b'][0],
    reference_conspiracy=BASELINE_CONSPIRACY,
    bootstrap_intervals=cox_bootstrap_results,
)

fig6 = plot_gateway_scatter(
    gateway_2d,
    cluster_assignments=cluster_assignments,
    figsize=NHB_COL_DOUBLE,
    reference_conspiracy=BASELINE_CONSPIRACY,
    text_scale=1.6,
    title='Contagiousness vs. Impact on Consecutive Adoption by Conspiracy',
    save_path=os.path.join(FIGURE_OPTIONAL_DIR, 'fig06_gateway_2d_scatter.png'),
)
plt.show()
print(f'Saved: {os.path.join(FIGURE_OPTIONAL_DIR, "fig06_gateway_2d_scatter.png")}')


---

## Figure 7 — Counterfactual Interventions (2 rows × 3 columns)

Three counterfactual interventions applied to the agent-based simulation, each compared against the baseline run:

- **Quarantine** — temporarily silence users who adopt a conspiracy for `t ∈ {4, 12, 24, 168}` hours.
- **Shadow banning** — drop a fraction of new adoptions according to a detection accuracy `r ∈ {0.5, 0.9, 0.95, 0.99, 1.0}`.
- **Reputation nudge** — reject a fraction of new adoptions with probability `r ∈ {0.01, 0.05, 0.10, 0.50, 1.0}`.

**Top row (a–c):** total AUC change vs baseline per intervention intensity. Blue bars = positive change in adoption, red bars = reduction. Error bars show 95\% simulation run bootstrap intervals. They represent Monte Carlo uncertainty only, not empirical sampling uncertainty.

**Bottom row (d–f):** efficiency — percent AUC reduction *per unit* of intervention intensity. Shows diminishing returns (or lack thereof) as intensity grows.

In [ ]:
# Figure 7 — Counterfactual intervention effects (2 rows × 3 columns).
# Top row: total AUC % change vs baseline per intervention intensity (bars).
# Bottom row: AUC % reduction per unit of intervention intensity (line).
# Self-contained: loads simulation_results_all.pkl, computes AUCs, then plots.
# Intervals describe Monte Carlo uncertainty from resampling simulation runs only.

import pandas as pd
from matplotlib.ticker import FixedLocator, FixedFormatter, NullLocator
from conspiracy_analysis.simulation.evaluation import compute_diffusion_curves

bundle = load_manifest_backed_simulation_bundle(intermediate_path('simulation_results_all.pkl'))
cfg = bundle['config']
counterfactual_steps = cfg['counterfactual_steps']


def _total_auc_by_run(results):
    """Total AUC summed across conspiracies for each ABM run."""
    df = compute_diffusion_curves(results, steps=counterfactual_steps)
    auc_records = []
    for (run, _consp), group in df.groupby(['RUN', 'conspiracy']):
        group = group.sort_values('t')
        auc = np.trapezoid(group['adoption_fraction'].values, group['t'].values)
        auc_records.append({'RUN': run, 'auc': auc})
    df_auc = pd.DataFrame(auc_records)
    return df_auc.groupby('RUN')['auc'].sum().sort_index()


baseline_auc = _total_auc_by_run(bundle['results_baseline'])
baseline_mean, baseline_ci_lower, baseline_ci_upper = bootstrap_mean_ci(baseline_auc.values)


def _paired_auc_summary(results, intensity):
    scenario_auc = _total_auc_by_run(results)
    common_runs = baseline_auc.index.intersection(scenario_auc.index)
    if len(common_runs) != len(baseline_auc) or len(common_runs) != len(scenario_auc):
        raise ValueError('Baseline and scenario run ids must match for paired bootstrap.')
    return bootstrap_auc_change_ci(
        baseline_auc.loc[common_runs].values,
        scenario_auc.loc[common_runs].values,
        intensity=intensity,
    )


def _collect_auc_summaries(results_list, intensities):
    summaries = [_paired_auc_summary(results, intensity) for results, intensity in zip(results_list, intensities)]
    changes = np.asarray([s['change'] for s in summaries], dtype=float)
    change_lowers = [s['change_ci'][0] for s in summaries]
    change_uppers = [s['change_ci'][1] for s in summaries]
    efficiencies = np.asarray([s['efficiency'] for s in summaries], dtype=float)
    efficiency_lowers = [s['efficiency_ci'][0] for s in summaries]
    efficiency_uppers = [s['efficiency_ci'][1] for s in summaries]
    return (
        changes,
        ci_error_array(changes, change_lowers, change_uppers),
        efficiencies,
        ci_error_array(efficiencies, efficiency_lowers, efficiency_uppers),
    )


# Quarantine: 4h, 12h, 24h, 168h (1 week)
q_labels = ['4 h', '12 h', '24 h', '7 d']
q_intensities = [4, 12, 24, 168]
q_changes, q_errors, q_eff, q_eff_err = _collect_auc_summaries(
    [
        bundle['results_quarantine_4h'],
        bundle['results_quarantine_12h'],
        bundle['results_quarantine_24h'],
        bundle['results_quarantine_168h'],
    ],
    q_intensities,
)

# Shadow banning: detection accuracy in [0.5, 1.0]
mod_rates = sorted(bundle['moderation_results'].keys())
mod_labels = [f'{int(r * 100)}%' for r in mod_rates]
mod_intensities = [r * 100 for r in mod_rates]
mod_changes, mod_errors, mod_eff, mod_eff_err = _collect_auc_summaries(
    [bundle['moderation_results'][r] for r in mod_rates],
    mod_intensities,
)

# Reputation nudge: rejection rate in [0.01, 1.0]
nudge_rates = sorted(bundle['nudge_results'].keys())
nudge_labels = [f'{int(r * 100)}%' for r in nudge_rates]
nudge_intensities = [r * 100 for r in nudge_rates]
nudge_changes, nudge_errors, nudge_eff, nudge_eff_err = _collect_auc_summaries(
    [bundle['nudge_results'][r] for r in nudge_rates],
    nudge_intensities,
)


def _bar_colors(values):
    return [NHB_PALETTE['positive'] if v > 0 else NHB_PALETTE['negative'] for v in values]


fig, axes = plt.subplots(
    2, 3,
    figsize=(NHB_COL_DOUBLE[0], NHB_COL_DOUBLE[1] * 1.15),
)

# ── Top row: total % AUC change vs baseline ──
top_panels = [
    (axes[0, 0], 'Quarantine',        'Quarantine Duration',
     q_labels,     q_changes,     q_errors,     'a'),
    (axes[0, 1], 'Shadow Banning',   'Detection Accuracy',
     mod_labels,   mod_changes,   mod_errors,   'b'),
    (axes[0, 2], 'Reputation Nudge', 'Rejection Rate',
     nudge_labels, nudge_changes, nudge_errors, 'c'),
]
for ax, title, xlabel, labels, values, errors, panel in top_panels:
    ax.axhline(0, color='#333333', linewidth=0.5)
    bars = ax.bar(
        labels, values, yerr=errors, capsize=3,
        color=_bar_colors(values), edgecolor='white', width=0.65,
    )
    for bar, val in zip(bars, values):
        offset = -3 if val < 0 else 1.5
        va = 'top' if val < 0 else 'bottom'
        ax.text(
            bar.get_x() + bar.get_width() / 2, val + offset,
            f'{val:+.1f}%', ha='center', va=va, fontsize=nhb_annotation_fontsize(),
        )
    ax.set_xlabel(xlabel)
    ax.set_title(title)
    ax.grid(axis='y', linestyle=':', linewidth=0.4, alpha=0.4)
    ax.set_axisbelow(True)
    ax.set_ylim(-100, 5)
    apply_panel_label(ax, panel)
axes[0, 0].set_ylabel('% AUC Change vs Baseline')

# ── Bottom row: efficiency (% AUC reduction per unit intensity) ──
# Use the same tick positions and labels as the top row so the x-axes line up.
bot_panels = [
    (axes[1, 0], 'Quarantine',        'Quarantine Duration',
     q_intensities,     q_eff,     q_eff_err,     q_labels,     'd', True),
    (axes[1, 1], 'Shadow Banning',   'Detection Accuracy',
     mod_intensities,   mod_eff,   mod_eff_err,   mod_labels,   'e', False),
    (axes[1, 2], 'Reputation Nudge', 'Rejection Rate',
     nudge_intensities, nudge_eff, nudge_eff_err, nudge_labels, 'f', False),
]
for ax, title, xlabel, x, y, y_err, tick_labels, panel, log_x in bot_panels:
    ax.axhline(0, color='#333333', linewidth=0.5)
    ax.errorbar(
        x, y, yerr=y_err,
        marker='o', capsize=3,
        color=NHB_PALETTE['positive'],
    )
    ax.set_xlabel(xlabel)
    ax.set_title(title)
    if log_x:
        ax.set_xscale('log')
        ax.xaxis.set_major_locator(FixedLocator(x))
        ax.xaxis.set_major_formatter(FixedFormatter(tick_labels))
        ax.xaxis.set_minor_locator(NullLocator())
    else:
        ax.set_xticks(x)
        ax.set_xticklabels(tick_labels)
    ax.grid(linestyle=':', linewidth=0.4, alpha=0.4)
    ax.set_axisbelow(True)
    apply_panel_label(ax, panel)
axes[1, 0].set_ylabel('% AUC Reduction per Unit Intensity')

fig.tight_layout()
# Pin both leftmost-column y-labels to the same axes-coordinate x so they
# stack vertically beneath each other regardless of per-row tick-label width.
for ax_left in axes[:, 0]:
    ax_left.yaxis.set_label_coords(-0.18, 0.5)
out_path = os.path.join(FIGURE_FINAL_DIR, 'fig07_counterfactual_composite.png')
fig.savefig(out_path, bbox_inches='tight')
plt.show()
print(f'Baseline total AUC = {baseline_mean:.1f} (95% simulation run bootstrap interval {baseline_ci_lower:.1f}, {baseline_ci_upper:.1f})')
print(f'Saved: {out_path}')


---

## Figure 8 — Combined Panel: Exposure HRs, Decay Times, Temporal Hazard Shift

A composite of the three core hazard-model panels for use as an alternative summary figure.

- **Panel a (top-left)** — exposure hazard ratios for first vs. second+ adoptions, re-referenced to one neighbor (mirrors Figure 2).
- **Panel b (bottom-left)** — analytical decay-to-baseline time $t^*$ by adoption depth (mirrors Figure 4).
- **Panel c (right, full height)** — instantaneous baseline hazard ratio (Weibull / Model 1) at 1 h, 24 h, and 72 h after the most recent adoption (mirrors Figure 3).

Per-panel titles are dropped and replaced with **a / b / c** labels; the descriptive content moves to the figure caption (matches the convention used in Figures 1 and 7).

In [ ]:
# Figure 8 — Combined panel: Fig 2 (top-left), Fig 4 (bottom-left), Fig 3 (right, full height).
# Inlines the matplotlib drawing from the three helper functions so the composite is
# self-contained in the notebook; reuses the underlying math / colour helpers from the
# library to avoid drift.
from matplotlib.transforms import ScaledTranslation

from conspiracy_analysis.visualization.figures import (
    _bootstrap_exposure_hrs, _MODEL_COLORS, _MODEL_LABELS,
)
from conspiracy_analysis.visualization.plots import _model_sort_key

# Shared color map between panel B (decay-time markers) and panel C (hazard-ratio bars)
# so each adoption-depth model uses the same colour in both panels.
panel_bc_colors = {
    'model_2a': '#E24A33', 'model_3':  '#348ABD', 'model_4':  '#988ED5',
    'model_5':  '#9467bd', 'model_6':  '#8EBA42', 'model_7':  '#FFB000',
    'model_8':  '#777777',
}
publication_hazard_models = [
    'model_2a', 'model_3', 'model_4', 'model_5',
    'model_6', 'model_7', 'model_8',
]
missing_hazard_models = [
    name for name in publication_hazard_models
    if name not in all_baselines or name not in decay_times
]
if missing_hazard_models:
    raise RuntimeError(
        f'Figure 8 requires fitted baseline and decay results for {missing_hazard_models}.'
    )


def _ordinal(n: int) -> str:
    """English ordinal: 1 -> '1st', 2 -> '2nd', 3 -> '3rd', 4 -> '4th', ..."""
    if 10 <= (n % 100) <= 20:
        suffix = 'th'
    else:
        suffix = {1: 'st', 2: 'nd', 3: 'rd'}.get(n % 10, 'th')
    return f'{n}{suffix}'


def _circle_panel_label(ax, letter: str,
                        dx_inches: float = -0.30,
                        dy_inches: float = 0.18) -> None:
    """Draw a bold panel letter inside a white circle at a FIXED pixel offset from
    the axes top-left corner. Using ScaledTranslation (inches) instead of axes
    fractions guarantees that circles in the same row/column line up uniformly
    across panels of different sizes (e.g. Fig 8's full-height panel C vs the
    half-height panels A and B).
    """
    fig = ax.figure
    transform = ax.transAxes + ScaledTranslation(
        dx_inches, dy_inches, fig.dpi_scale_trans,
    )
    ax.text(
        0, 1, letter,
        transform=transform,
        fontsize=16, fontweight='bold',
        ha='center', va='center',
        bbox=dict(
            boxstyle='circle,pad=0.4',
            facecolor='white',
            edgecolor='black',
            linewidth=1.5,
        ),
        zorder=10,
        clip_on=False,
    )


# Extra height + manual top/hspace margin so the panel-label circles sit cleanly
# above each subplot without being clipped.
fig, axd = plt.subplot_mosaic(
    [['A', 'C'],
     ['B', 'C']],
    figsize=(NHB_COL_DOUBLE[0], NHB_COL_DOUBLE[1] * 1.15),
)

# ── Panel A (top-left): Exposure HRs, first vs second+ adoption (mirrors Figure 2) ──
ax = axd['A']
panel_a_models = {
    'First Conspiracy Theory Adoption':   stratified_models['model_1_strat'],
    'Second+ Conspiracy Theory Adoption': stratified_models['model_pooled'],
}
exposure_vars = ['s7_d2', 's7_d3', 's7_d4']
exposure_labels = ['2', '3', '4+']
xa = np.arange(len(exposure_vars))
n_a = len(panel_a_models)
jitter_width = 0.06
offsets = (np.arange(n_a) - (n_a - 1) / 2) * jitter_width

for m_idx, label in enumerate(panel_a_models):
    boot_model_name = 'model_1_strat' if label.startswith('First') else 'model_pooled'
    boot_values = _bootstrap_exposure_hrs(boot_model_name, exposure_vars, cox_bootstrap_results)
    if boot_values is None:
        raise RuntimeError(
            f'Figure 8 requires bootstrap exposure intervals for {boot_model_name}.'
        )
    hrs, ci_lowers, ci_uppers = boot_values
    yerr = np.vstack([hrs - ci_lowers, ci_uppers - hrs])
    yerr = np.clip(yerr, 0, None)
    color = _MODEL_COLORS.get(label, f'C{m_idx}')
    legend_label = _MODEL_LABELS.get(label, label)
    ax.errorbar(
        xa + offsets[m_idx], hrs, yerr=yerr,
        fmt='o-', color=color, markersize=8, linewidth=2,
        capsize=5, capthick=1.5, label=legend_label,
    )

ax.axhline(1.0, color='grey', linestyle='--', linewidth=0.8, zorder=0)
ax.set_xticks(xa)
ax.set_xticklabels(exposure_labels)
ax.set_xlabel('Number of Conspiracy-Sharing Neighbors (14-day window)')
ax.set_ylabel('Hazard Ratio (ref = 1 neighbor)')
ax.legend(loc='upper left')

# ── Panel B (bottom-left): Decay-to-baseline times by adoption depth (mirrors Figure 4) ──
# X-axis tick labels and axis label match Panel C's "After Nth Conspiracy" legend so the
# reader can connect each marker to the corresponding bar group on the right.
# Mapping: model_2a -> after 1st adoption, model_3 -> after 2nd, ...
ax = axd['B']
names_b = sorted(publication_hazard_models, key=_model_sort_key)
boot_decay = cox_bootstrap_results['decay_time_intervals'].set_index('model')
values_days = []
lower_days = []
upper_days = []
for n in names_b:
    if n in boot_decay.index:
        values_days.append(boot_decay.loc[n, 't_star_hours'] / 24)
        lower_days.append(boot_decay.loc[n, 'ci_lower'] / 24)
        upper_days.append(boot_decay.loc[n, 'ci_upper'] / 24)
    else:
        raise RuntimeError(
            f'Figure 8 requires a bootstrap decay interval for {n}.'
        )
xb_labels = []
for n in names_b:
    digit = int(''.join(c for c in n.replace('model_', '') if c.isdigit()))
    # model_N corresponds to "after (N-1)th adoption"; model_2a is the same as N=2.
    xb_labels.append(_ordinal(digit - 1) if digit > 1 else 'Baseline')
xb = np.arange(len(names_b))
ax.plot(xb, values_days, '-', color='lightgrey', linewidth=2, zorder=1)
ax.errorbar(
    xb, values_days,
    yerr=np.vstack([
        np.maximum(np.asarray(values_days) - np.asarray(lower_days), 0.0),
        np.maximum(np.asarray(upper_days) - np.asarray(values_days), 0.0),
    ]),
    fmt='none', capsize=4, color='black', elinewidth=0.8, capthick=0.8, zorder=1,
)
for i, (name, val) in enumerate(zip(names_b, values_days)):
    if not np.isfinite(val):
        continue
    ax.plot(
        i, val, marker='o', markersize=14,
        color=panel_bc_colors.get(name, 'steelblue'),
        markeredgecolor='black', markeredgewidth=0.7, zorder=2,
    )
    ax.annotate(
        f'{val:.1f}d', xy=(i, val),
        xytext=(0, 12), textcoords='offset points',
        ha='center', fontsize=nhb_annotation_fontsize(),
    )
ax.set_xticks(xb)
ax.set_xticklabels(xb_labels)
ax.set_xlabel('After Nth Conspiracy Adoption')
ax.set_ylabel('Time for hazard to recover to baseline (days)')
ax.grid(axis='y', linestyle='--', alpha=0.3)

# ── Panel C (right, full height): Temporal hazard shift at 1h / 24h / 72h (mirrors Figure 3) ──
ax = axd['C']
eval_times_hours = [1.0, 24.0, 72.0]
weibull_models_c = publication_hazard_models
panel_c_ratios = {}
for name in weibull_models_c:
    boot_temporal = cox_bootstrap_results['temporal_hazard_ratio_intervals']
    ratios = []
    lowers = []
    uppers = []
    for t in eval_times_hours:
        row = boot_temporal[
            (boot_temporal['model'] == name)
            & (np.isclose(boot_temporal['time_hours'].astype(float), float(t)))
        ]
        if row.empty:
            raise RuntimeError(
                f'Figure 8 requires a bootstrap temporal interval for {name} at {t} hours.'
            )
        ratios.append(float(row['ratio'].iloc[0]))
        lowers.append(float(row['ci_lower'].iloc[0]))
        uppers.append(float(row['ci_upper'].iloc[0]))
    panel_c_ratios[name] = ratios
    panel_c_lowers = locals().setdefault('panel_c_lowers', {})
    panel_c_uppers = locals().setdefault('panel_c_uppers', {})
    panel_c_lowers[name] = lowers
    panel_c_uppers[name] = uppers

panel_c_names = {
    'model_2a': 'After 1st Conspiracy', 'model_3':  'After 2nd Conspiracy',
    'model_4':  'After 3rd Conspiracy', 'model_5':  'After 4th Conspiracy',
    'model_6':  'After 5th Conspiracy', 'model_7':  'After 6th Conspiracy',
    'model_8':  'After 7th Conspiracy',
}

xc_labels = [f'{t:.0f}h' if t >= 1 else f'{t*60:.0f}m' for t in eval_times_hours]
xc = np.arange(len(xc_labels))
nc = len(panel_c_ratios)
width = 0.8 / max(nc, 1)

for i, (name, ratios) in enumerate(panel_c_ratios.items()):
    offset = (i - (nc - 1) / 2) * width
    bars = ax.bar(
        xc + offset, ratios, width,
        yerr=np.vstack([
            np.maximum(np.asarray(ratios) - np.asarray(panel_c_lowers[name]), 0.0),
            np.maximum(np.asarray(panel_c_uppers[name]) - np.asarray(ratios), 0.0),
        ]),
        capsize=2,
        error_kw=dict(ecolor='black', elinewidth=0.8, capthick=0.8),
        color=panel_bc_colors.get(name, f'C{i}'),
        edgecolor='black', alpha=0.85,
        label=panel_c_names.get(name, name),
    )
    # Per-bar value labels for Panel (c) - commented out per request;
    # uncomment the block below to restore the 'N.Nx' annotations above each bar.
    # for bar, ratio in zip(bars, ratios):
    #     ax.annotate(
    #         f'{ratio:.1f}×',
    #         xy=(bar.get_x() + bar.get_width() / 2, bar.get_height()),
    #         xytext=(0, 5), textcoords='offset points',
    #         ha='center', fontsize=nhb_annotation_fontsize(),
    #     )

ax.set_xticks(xc)
ax.set_xticklabels(xc_labels)
ax.axhline(1.0, color='grey', linestyle='--', linewidth=1.2,
           label='Before 1st Conspiracy (Baseline)')
ax.set_xlabel('Time Since Last Conspiracy Adoption (hours)')
ax.set_ylabel('Baseline Hazard Ratio (vs. Before 1st Conspiracy)')
ax.legend()

fig.tight_layout()
# Add explicit top margin and inter-row spacing so the circle labels above panels A and
# B are not clipped at the figure top / between rows. Apply BEFORE drawing the labels
# so each ax.transAxes references its final position.
fig.subplots_adjust(top=0.82, hspace=0.55)

# Panel labels — bold letters inside white circles, placed at a uniform pixel offset
# from each axes' top-left corner so they line up across panels.
_circle_panel_label(axd['A'], 'a', dy_inches=0.3)
_circle_panel_label(axd['B'], 'b', dy_inches=0.7)
_circle_panel_label(axd['C'], 'c', dy_inches=0.3)

out_path = os.path.join(FIGURE_FINAL_DIR, 'fig08_combined_hazard_panel.png')
fig.savefig(out_path, dpi=300, bbox_inches='tight', pad_inches=0.2)
plt.show()
print(f'Saved: {out_path}')

---

## Figure 9 — Combined Panel: Settler Effect + Gateway Scatter

A composite of Figures 5 and 6.

- **Panels a–c (top row)** — settler-effect pattern under three clusterings: semantic, first-appearance, and peak-frequency (mirrors Figure 5).
- **Panel d (bottom, full width)** — 2D gateway scatter: per-conspiracy contagiousness (Model 1 HR) on the x-axis vs gateway acceleration (Model 2b HR) on the y-axis, both relative to *fakenews* (mirrors Figure 6).

Per-panel descriptive titles are kept on the top row (one per clustering) and dropped on the bottom panel; the descriptive content for panel d moves to the figure caption. **a / b / c / d** labels follow the same NHB convention as Figures 1, 7, and 8.

In [ ]:
# Figure 9 — Combined panel: Fig 5 (top row, three settler panels) and Fig 6 (bottom, gateway scatter).
# Top row reuses _settler_on_ax directly so the per-panel drawing matches Figure 5;
# bottom row inlines the gateway-scatter body so the composite is self-contained.
from matplotlib.transforms import ScaledTranslation

from conspiracy_analysis.visualization.figures import _settler_on_ax
from conspiracy_analysis.visualization.plots import _CONSPIRACY_LABELS
from conspiracy_analysis.models.gateway_effects import identify_gateway_2d
from conspiracy_analysis.analysis.semantic import build_cluster_display_metadata

GATEWAY_CLUSTER_LABELS, GATEWAY_CLUSTER_COLORS = build_cluster_display_metadata(
    cluster_assignments
)

# Font sizes — bumped for legibility at the composite scale.
SETTLER_TEXT_SIZE = 13   # significance markers, in-panel annotations
SETTLER_TITLE_SIZE = 14
GATEWAY_LABEL_SIZE = 13  # per-point labels and reference annotation
GATEWAY_AXIS_SIZE = 13


def _circle_panel_label(ax, letter: str,
                        dx_inches: float = -0.30,
                        dy_inches: float = 0.30) -> None:
    """Draw a bold panel letter inside a white circle at a FIXED pixel offset from
    the axes top-left corner. Using ScaledTranslation (inches) instead of axes
    fractions guarantees that circles in the same row/column line up uniformly
    across panels of different sizes (e.g. Fig 9's tall panel D vs the short
    settler panels A/B/C).
    """
    fig = ax.figure
    transform = ax.transAxes + ScaledTranslation(
        dx_inches, dy_inches, fig.dpi_scale_trans,
    )
    ax.text(
        0, 1, letter,
        transform=transform,
        fontsize=16, fontweight='bold',
        ha='center', va='center',
        bbox=dict(
            boxstyle='circle,pad=0.4',
            facecolor='white',
            edgecolor='black',
            linewidth=1.5,
        ),
        zorder=10,
        clip_on=False,
    )


def _find_bracket_pairs(ax):
    """Find significance bracket lines + their label texts on a settler axes.

    `_settler_on_ax` draws each bracket as a 4-point line with ydata of the
    form [y - tick, y, y, y - tick] (linewidth=1.0) and a sibling Text artist
    for the p-value label. Returned pairs are sorted by line top-y ascending.
    """
    bracket_lines = []
    for ln in ax.lines:
        yd = ln.get_ydata()
        if (len(yd) == 4 and yd[0] == yd[3] and yd[1] == yd[2] and yd[0] < yd[1]):
            bracket_lines.append(ln)
    n = len(bracket_lines)
    if n == 0:
        return []
    bracket_texts = list(ax.texts[-n:])  # the helper draws texts last, in order
    return sorted(
        zip(bracket_lines, bracket_texts),
        key=lambda p: p[0].get_ydata()[1],
    )


def _harmonize_bracket_gap(axes, vertical_gap: float) -> float:
    """Force the gap between the lowest bracket and the topmost bracket to be
    `vertical_gap` (in data y-units) on every panel, so the within-panel
    spacing reads the same regardless of each panel's data span.

    Returns the highest absolute y-coordinate any bracket label ends up at,
    so the caller can extend the y-axis limit to fit it.
    """
    max_top = -np.inf
    for ax in axes:
        pairs = _find_bracket_pairs(ax)
        if len(pairs) < 2:
            if pairs:
                max_top = max(max_top, pairs[-1][0].get_ydata()[1])
            continue
        anchor_y = pairs[0][0].get_ydata()[1]  # lowest bracket
        target_top_y = anchor_y + vertical_gap
        top_line, top_text = pairs[-1]
        delta = target_top_y - top_line.get_ydata()[1]
        if delta != 0:
            top_line.set_ydata([y + delta for y in top_line.get_ydata()])
            tx, ty = top_text.get_position()
            top_text.set_position((tx, ty + delta))
        max_top = max(max_top, target_top_y)
    return max_top


fig, axd = plt.subplot_mosaic(
    [['A', 'B', 'C'],
     ['D', 'D', 'D']],
    figsize=(NHB_COL_DOUBLE[0] * 1.1, NHB_COL_DOUBLE[1] * 1.85),
    gridspec_kw={'height_ratios': [1, 2]},
)

# ── Panels a / b / c (top row): Settler effect under three clusterings (mirrors Figure 5) ──
settler_panels = [
    ('A', 'Semantic clustering',         semantic_settler),
    ('B', 'First-appearance clustering', temporal_settler),
    ('C', 'Peak-frequency clustering',   peak_freq_settler),
]
for key, title, settler in settler_panels:
    ax = axd[key]
    _settler_on_ax(
        ax, settler, n_boot=2000, show_significance=True,
        bootstrap_intervals=timeline_bootstrap_results,
        bootstrap_family={'A': 'semantic', 'B': 'temporal', 'C': 'peak_frequency'}[key],
    )
    ax.set_title(title, fontsize=SETTLER_TITLE_SIZE, pad=8)
    # Bump significance bracket text + any in-panel annotations the helper drew.
    for txt in ax.texts:
        txt.set_fontsize(SETTLER_TEXT_SIZE)
    ax.tick_params(axis='both', labelsize=11)
    ax.xaxis.label.set_fontsize(GATEWAY_AXIS_SIZE)
    ax.yaxis.label.set_fontsize(GATEWAY_AXIS_SIZE)

# Share y first so the bracket gap can be computed from the shared y-range —
# otherwise each panel's per-data step is wildly different (b's data span is much
# smaller than a's / c's, which is why the helper produced uneven bracket gaps).
top_settler_axes = [axd[k] for k, _, _ in settler_panels]
top_y = max(ax.get_ylim()[1] for ax in top_settler_axes)
for ax in top_settler_axes:
    ax.set_ylim(top=top_y * 1.18)

# Now harmonize: every panel gets the same absolute vertical gap between its
# lowest and topmost brackets, derived from the shared y-axis span.
shared_y_min, shared_y_max = top_settler_axes[0].get_ylim()
shared_span = shared_y_max - shared_y_min
uniform_gap = shared_span * 0.10  # ~10% of the panel height
new_top = _harmonize_bracket_gap(top_settler_axes, uniform_gap)
# Extend ylim if the harmonized top bracket would clip.
if new_top * 1.04 > shared_y_max:
    for ax in top_settler_axes:
        ax.set_ylim(top=new_top * 1.06)

for ax in top_settler_axes[1:]:
    ax.set_ylabel('')

# ── Panel d (bottom, full width): 2D gateway scatter (mirrors Figure 6) ──
ax = axd['D']
gateway_boot = cox_bootstrap_results.get('gateway_intervals')
if not isinstance(gateway_boot, pd.DataFrame) or gateway_boot.empty:
    raise RuntimeError('Figure 9 requires bootstrap gateway intervals.')
required_gateway_columns = {
    'conspiracy', 'model1_hr', 'model1_ci_lower', 'model1_ci_upper',
    'model2_hr', 'model2_ci_lower', 'model2_ci_upper', 'interval_source',
}
missing_gateway_columns = sorted(required_gateway_columns.difference(gateway_boot.columns))
if missing_gateway_columns:
    raise RuntimeError(
        f'Figure 9 bootstrap gateway intervals lack columns: {missing_gateway_columns}.'
    )
gateway_nonreference = gateway_boot[gateway_boot['interval_source'] != 'reference']
if gateway_nonreference.empty or not gateway_nonreference['interval_source'].eq('bootstrap').all():
    raise RuntimeError('Figure 9 requires bootstrap intervals for every gateway point.')
expected_gateway_conspiracies = {
    name.replace('ConsProb_', '', 1)
    for name in INCLUDED_CONSPIRACIES
    if name != BASELINE_CONSPIRACY
}
actual_gateway_conspiracies = set(gateway_nonreference['conspiracy'].astype(str))
if actual_gateway_conspiracies != expected_gateway_conspiracies:
    missing_gateway = sorted(expected_gateway_conspiracies.difference(actual_gateway_conspiracies))
    unexpected_gateway = sorted(actual_gateway_conspiracies.difference(expected_gateway_conspiracies))
    raise RuntimeError(
        f'Figure 9 bootstrap gateway rows differ from the configured conspiracies. '
        f'Missing: {missing_gateway}. Unexpected: {unexpected_gateway}.'
    )
gateway_interval_columns = [
    'model1_hr', 'model1_ci_lower', 'model1_ci_upper',
    'model2_hr', 'model2_ci_lower', 'model2_ci_upper',
]
if gateway_nonreference[gateway_interval_columns].isna().any().any():
    raise RuntimeError('Figure 9 bootstrap gateway intervals contain missing values.')

gateway_2d = identify_gateway_2d(
    cox_results['model_1'][0],
    cox_results['model_2b'][0],
    reference_conspiracy=BASELINE_CONSPIRACY,
    bootstrap_intervals=cox_bootstrap_results,
)

df_g = gateway_2d.copy()
df_g['cluster'] = df_g['conspiracy'].map(GATEWAY_CLUSTER_LABELS).fillna('Unknown')
df_g['label']   = df_g['conspiracy'].map(_CONSPIRACY_LABELS).fillna(df_g['conspiracy'])

ref_short = BASELINE_CONSPIRACY.replace('ConsProb_', '')
ref_label = _CONSPIRACY_LABELS.get(ref_short, ref_short)
df_g = df_g[df_g['conspiracy'] != ref_short]

for cluster_name, color in GATEWAY_CLUSTER_COLORS.items():
    sub = df_g[df_g['cluster'] == cluster_name]
    if sub.empty:
        continue
    xerr = np.vstack([
        (sub['model1_hr'] - sub['model1_ci_lower']).values,
        (sub['model1_ci_upper'] - sub['model1_hr']).values,
    ])
    yerr = np.vstack([
        (sub['model2_hr'] - sub['model2_ci_lower']).values,
        (sub['model2_ci_upper'] - sub['model2_hr']).values,
    ])
    ax.errorbar(
        sub['model1_hr'], sub['model2_hr'],
        xerr=xerr, yerr=yerr,
        fmt='o', color=color, markersize=8,
        capsize=3, elinewidth=1, alpha=0.85,
        label=cluster_name,
    )

ax.axhline(1.0, color='grey', linestyle='--', linewidth=0.8, zorder=0)
ax.axvline(1.0, color='grey', linestyle='--', linewidth=0.8, zorder=0)
ax.annotate(
    f'{ref_label}\n(reference = 1, 1)',
    xy=(1.0, 1.0), xytext=(0.9, 1.1),
    textcoords='data',
    fontsize=GATEWAY_LABEL_SIZE, fontstyle='italic', color='grey', ha='left',
    arrowprops=dict(arrowstyle='->', color='grey', lw=0.8),
)
for _, row in df_g.iterrows():
    ax.annotate(
        row['label'],
        xy=(row['model1_hr'], row['model2_hr']),
        xytext=(5, 5), textcoords='offset points',
        fontsize=GATEWAY_LABEL_SIZE, ha='left', va='bottom',
    )

ax.set_xlabel(
    f'Contagiousness (HR of First Adoption, ref = {ref_label})',
    fontsize=GATEWAY_AXIS_SIZE,
)
ax.set_ylabel(
    f'Impact on Consecutive Adoption (Gateway Effect, ref = {ref_label})',
    fontsize=GATEWAY_AXIS_SIZE,
)
ax.tick_params(axis='both', labelsize=11)
legend = ax.legend(
    title='Semantic Cluster', loc='upper center',
    framealpha=0.9, fontsize=GATEWAY_LABEL_SIZE,
)
if legend is not None and legend.get_title() is not None:
    legend.get_title().set_fontsize(GATEWAY_LABEL_SIZE)

fig.tight_layout()
# Extra top margin + inter-row whitespace so the circle labels above the settler
# row aren't clipped and don't crowd the bottom panel. Apply BEFORE drawing the
# labels so each ax.transAxes references its final position.
fig.subplots_adjust(top=0.82, hspace=0.25)

# Panel labels — bold letters inside white circles, placed at a uniform pixel offset
# from each axes' top-left corner. Every circle uses the same offset, so circles in
# the same row line up vertically and circles in the same column line up horizontally.
_circle_panel_label(axd['A'], 'a')
_circle_panel_label(axd['B'], 'b')
_circle_panel_label(axd['C'], 'c')
_circle_panel_label(axd['D'], 'd')

out_path = os.path.join(FIGURE_FINAL_DIR, 'fig09_settler_gateway_combined.png')
fig.savefig(out_path, dpi=300, bbox_inches='tight', pad_inches=0.2)
plt.show()
print(f'Saved: {out_path}')
